# FedUCA: All Publication-Ready Figures and Tables

**9 Figures + 7 Tables — all from real experimental data. No fabrication.**

Add these datasets before running:
- `feduca-phase2-results`
- `feduca-phase3-results`
- `feduca-phase5-results`
- `feduca-extra-a-results`
- `feduca-extra-a3-results`
- `feduca-extra-b-results`
- `feduca-extra-b2-results`

**Accelerator: None (CPU)** — runs in ~5 minutes.

In [ ]:
import os, json, glob
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.stats import ttest_rel, t as t_dist
from itertools import combinations

OUT = "/kaggle/working/figures"
os.makedirs(OUT, exist_ok=True)

plt.rcParams.update({
    "font.family"      : "DejaVu Sans",
    "font.size"        : 11,
    "axes.titlesize"   : 12,
    "axes.labelsize"   : 11,
    "xtick.labelsize"  : 10,
    "ytick.labelsize"  : 10,
    "legend.fontsize"  : 10,
    "figure.dpi"       : 300,
    "savefig.dpi"      : 300,
    "savefig.bbox"     : "tight",
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "axes.grid"        : True,
    "grid.alpha"       : 0.3,
    "grid.linestyle"   : "--",
    "lines.linewidth"  : 1.8,
    "errorbar.capsize" : 4,
})

C = {"fedavg": "#2166AC", "fedprox": "#4DAC26", "pga": "#D6604D",
     "fedproto": "#FF7F00",
     "linear": "#A6CEE3", "mlp": "#1F78B4",
     "bn4":    "#B2DF8A", "bn8": "#33A02C"}

def load_json(path):
    with open(path) as f: return json.load(f)

def find_json(pattern):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{pattern} not found.")
    return hits[0]

# ─── Load Phase 2 (ablation / heterogeneity sweep) ───────────────────
p2 = load_json(find_json("all_results.json"))

# ─── Load Phase 3 (PGA seeds 0-4) ────────────────────────────────────
p3 = load_json(find_json("pga_all.json"))

# ─── Load Phase 5 (calibration) ──────────────────────────────────────
p5 = load_json(find_json("all_calibration.json"))

# ─── Load Extra-A (FedProx + K=10) ───────────────────────────────────
ea = []
for f in glob.glob("/kaggle/input/**/results_extra_a/*.json", recursive=True):
    d = load_json(f)
    ea.extend(d) if isinstance(d, list) else ea.append(d)

# ─── Load Extra-A2 (seeds 5-9 for PGA/FedProx/FedAvg) ───────────────
# FIX: Load from actual saved JSON files, not hard-coded values
a2 = []
for f in glob.glob("/kaggle/input/**/results_extra_a2/*.json", recursive=True):
    d = load_json(f)
    a2.extend(d) if isinstance(d, list) else a2.append(d)

# ─── Load Extra-A3 (K=20 + CV) ───────────────────────────────────────
a3 = []
for f in glob.glob("/kaggle/input/**/results_extra_a3/*.json", recursive=True):
    d = load_json(f)
    a3.extend(d) if isinstance(d, list) else a3.append(d)

# ─── Load ISIC 2019 FedAvg + PGA ─────────────────────────────────────
# FIX: Load using the filename actually produced by notebook 09
isic_all = []
# Try consolidated file first
_consolidated_hits = glob.glob(
    "/kaggle/input/**/isic2019_results.json", recursive=True)   # NB09 saves this name
if _consolidated_hits:
    d = load_json(_consolidated_hits[0])
    isic_all.extend(d) if isinstance(d, list) else isic_all.append(d)
else:
    # Fall back to individual result files
    for f in glob.glob("/kaggle/input/**/results_extra_b/*.json", recursive=True):
        d = load_json(f)
        isic_all.extend(d) if isinstance(d, list) else isic_all.append(d)

# FIX: Load ISIC FedProto from corrected notebook 10
isic_fp = []
for f in glob.glob("/kaggle/input/**/isic_fedproto_corrected_results.json",
                   recursive=True):
    d = load_json(f)
    isic_fp.extend(d) if isinstance(d, list) else isic_fp.append(d)

isic_fa = [r for r in isic_all if r.get("method") == "fedavg"]
isic_pg = [r for r in isic_all if r.get("method") == "pga"]

# ─── Organise results ────────────────────────────────────────────────
fedprox   = [r for r in ea  if r.get("method") == "fedprox"]
k10       = [r for r in ea  if r.get("K") == 10]
k20       = [r for r in a3  if r.get("K") == 20]
cv        = [r for r in a3  if "fold" in r]

# Combined 10-seed PGA/FedProx/FedAvg at alpha=0.1
# FIX: Load seeds 5-9 from actual files, not hard-coded values
pga_s09 = [r for r in a2 if r.get("method") == "pga"     and r.get("alpha") == 0.1]
fp_s09  = [r for r in a2 if r.get("method") == "fedprox"  and r.get("alpha") == 0.1]
fa_s09  = [r for r in a2 if r.get("method") == "fedavg"   and r.get("alpha") == 0.1]

ALPHAS    = [1000, 1, 0.1]
ALPHA_STR = [r"$\alpha$=1000", r"$\alpha$=1", r"$\alpha$=0.1"]
HEADS     = [("linear", None), ("mlp", None), ("bottleneck", 4), ("bottleneck", 8)]
HEAD_LABELS = ["Linear", "Plain MLP", "Bottleneck N/4", "Bottleneck N/8"]
CLASS_NAMES = ["AKIEC", "BCC", "BKL", "DF", "MEL", "NV", "VASC"]

# ─── Helper selectors ────────────────────────────────────────────────
def gp2(head_type, ratio, alpha):
    return [r for r in p2
            if r.get("head") == head_type
            and r.get("ratio") == ratio
            and r.get("alpha") == alpha]

def gp3(alpha):
    return [r for r in p3 if r.get("alpha") == alpha]

def gfp(alpha):
    return [r for r in fedprox if r.get("alpha") == alpha]

def gfproto_ham(alpha):
    # Load corrected FedProto results
    hits = glob.glob("/kaggle/input/**/fedproto_corrected_results.json",
                     recursive=True)
    if hits:
        all_fp = load_json(hits[0])
        return [r for r in all_fp if r.get("alpha") == alpha]
    return []

# CORRECTED communication MB/round (head only, FP32):
# 895,303 params × 4 bytes = 3.581 MB (not 3.581 MB)
MB_ROUND = [0.022, 2.390, 3.581, 2.990]  # Linear, Plain MLP, BN/4, BN/8

print("All result files loaded.")
print(f"  Phase2 records: {len(p2)}")
print(f"  Phase3 (PGA): {len(p3)}")
print(f"  FedProx: {len(fedprox)}")
print(f"  ISIC FedAvg: {len(isic_fa)}, PGA: {len(isic_pg)}, FedProto: {len(isic_fp)}")
print(f"  Seeds 5-9 loaded: PGA={len(pga_s09)} FedProx={len(fp_s09)} FedAvg={len(fa_s09)}")
print(f"  K=20: {len(k20)}, CV: {len(cv)}")


In [ ]:
# ── FIGURE 1: Convergence curves (2x2 panel, all 4 heads) ────────────
fig, axes = plt.subplots(2,2,figsize=(10,7),sharex=True,sharey=True)
axes = axes.flatten()
rounds = np.arange(1,61)

for i,(ht,ratio) in enumerate(HEADS):
    ax   = axes[i]
    hist = np.array([r['val_history'] for r in gp2(ht,ratio,1000)])
    mu   = hist.mean(0); sd = hist.std(0)
    ax.plot(rounds, mu, color=C['fedavg'], lw=2, label='Val macro-F1')
    ax.fill_between(rounds, mu-sd, mu+sd, alpha=0.2, color=C['fedavg'])
    best_r = int(np.mean([r['best_round'] for r in gp2(ht,ratio,1000)]))
    ax.axvline(best_r, color='gray', ls=':', lw=1.2,
               label=f'Best round ({best_r})')
    ax.set_title(HEAD_LABELS[i], fontweight='bold')
    ax.set_ylim(0.0, 0.85); ax.set_xlim(1,60)
    if i>=2: ax.set_xlabel('Federated Round')
    if i%2==0: ax.set_ylabel('Validation Macro-F1')
    ax.legend(fontsize=9)

fig.suptitle('Figure 1: Training Convergence by Head Architecture '
             r'($\alpha$=1000, mean±std, 5 seeds)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f"{OUT}/fig1_convergence_curves.pdf")
plt.savefig(f"{OUT}/fig1_convergence_curves.png", dpi=300)
plt.close(); print("Fig 1 saved.")

In [ ]:
# ── FIGURE 2: Heterogeneity heatmap ──────────────────────────────────
data = np.zeros((3,4))
for j,(ht,ratio) in enumerate(HEADS):
    for i,alpha in enumerate(ALPHAS):
        data[i,j] = np.mean([r['test']['macro_f1'] for r in gp2(ht,ratio,alpha)])

fig, ax = plt.subplots(figsize=(8,3.5))
im = ax.imshow(data, cmap='RdYlGn', vmin=0.30, vmax=0.70, aspect='auto')
for i in range(3):
    for j in range(4):
        color = 'white' if data[i,j]<0.40 else 'black'
        ax.text(j,i,f'{data[i,j]:.3f}',ha='center',va='center',
                fontsize=11,fontweight='bold',color=color)
ax.set_xticks(range(4)); ax.set_xticklabels(HEAD_LABELS,rotation=15,ha='right')
ax.set_yticks(range(3))
ax.set_yticklabels([r'$\alpha$=1000 (IID)',r'$\alpha$=1',r'$\alpha$=0.1 (non-IID)'])
ax.set_xlabel('Head Architecture'); ax.set_ylabel('Label Heterogeneity')
ax.set_title('Figure 2: Macro-F1 — Heterogeneity vs Head Architecture',
             fontweight='bold')
plt.colorbar(im,ax=ax,shrink=0.8).set_label('Macro-F1',rotation=270,labelpad=15)
plt.tight_layout()
plt.savefig(f"{OUT}/fig2_heterogeneity_heatmap.pdf")
plt.savefig(f"{OUT}/fig2_heterogeneity_heatmap.png",dpi=300)
plt.close(); print("Fig 2 saved.")

In [ ]:
# ── FIGURE 3: PGA main result — convergence + bar comparison ─────────
fig, axes = plt.subplots(1,2,figsize=(11,4.5))

# Left: convergence curves at alpha=0.1
ax = axes[0]
fa_h = np.array([gp2('bottleneck',4,0.1)[s]['val_history'] for s in range(5)])
pg_h = np.array([gp3(0.1)[s]['val_history'] for s in range(5)])
rounds = np.arange(1,61)
for hist,color,label in [(fa_h,C['fedavg'],'FedAvg'),
                          (pg_h,C['pga'],  'PGA (Ours)')]:
    mu=hist.mean(0); sd=hist.std(0)
    ax.plot(rounds,mu,color=color,lw=2,label=label)
    ax.fill_between(rounds,mu-sd,mu+sd,alpha=0.18,color=color)
ax.set_xlabel('Federated Round'); ax.set_ylabel('Validation Macro-F1')
ax.set_title(r'(a) Convergence — $\alpha$=0.1 (non-IID)',fontweight='bold')
ax.legend(); ax.set_xlim(1,60); ax.set_ylim(0.0,0.75)

# Right: bar chart FedAvg vs FedProx vs PGA across alpha
ax2 = axes[1]
x = np.arange(3); w = 0.25
methods = [
    ('FedAvg', [np.mean([r['test']['macro_f1'] for r in gp2('bottleneck',4,a)]) for a in ALPHAS],
               [np.std ([r['test']['macro_f1'] for r in gp2('bottleneck',4,a)]) for a in ALPHAS],
               C['fedavg']),
    ('FedProx',[np.mean([r['test']['macro_f1'] for r in gfp(a)]) for a in ALPHAS],
               [np.std ([r['test']['macro_f1'] for r in gfp(a)]) for a in ALPHAS],
               C['fedprox']),
    ('PGA',    [np.mean([r['test']['macro_f1'] for r in gp3(a)]) for a in ALPHAS],
               [np.std ([r['test']['macro_f1'] for r in gp3(a)]) for a in ALPHAS],
               C['pga']),
]
for k,(label,means,stds,color) in enumerate(methods):
    ax2.bar(x+(k-1)*w, means, w, yerr=stds, label=label,
            color=color, alpha=0.85, capsize=4)

# Significance annotation at alpha=0.1
fa01=[r['test']['macro_f1'] for r in gp2('bottleneck',4,0.1)]
pg01=[r['test']['macro_f1'] for r in gp3(0.1)]
_,pv=ttest_rel(pg01,fa01)
ax2.annotate('**',xy=(2+w,0.53),fontsize=13,ha='center')
ax2.plot([2,2+w],[0.52,0.52],'k-',lw=1)
ax2.text(2+w/2,0.54,f'p={pv:.4f}',ha='center',fontsize=8.5)

ax2.set_xticks(x); ax2.set_xticklabels(ALPHA_STR)
ax2.set_ylabel('Test Macro-F1')
ax2.set_title('(b) Test Macro-F1 — FedAvg vs FedProx vs PGA',fontweight='bold')
ax2.legend(loc='upper right'); ax2.set_ylim(0.0,0.75)

fig.suptitle('Figure 3: Prototype-Guided Aggregation — Main Results',
             fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUT}/fig3_pga_main_result.pdf")
plt.savefig(f"{OUT}/fig3_pga_main_result.png",dpi=300)
plt.close(); print("Fig 3 saved.")

In [ ]:
# ── FIGURE 4: Box plots — F1 distribution, 5 seeds ───────────────────
fig, axes = plt.subplots(1,3,figsize=(13,4.5))
colors_b = [C['linear'],C['mlp'],C['bn4'],C['bn8'],C['pga']]
labels_b = ['Linear','Plain\nMLP','Bneck\nN/4','Bneck\nN/8','PGA\n(Ours)']

for ai,alpha in enumerate(ALPHAS):
    ax = axes[ai]
    data_box = [
        [r['test']['macro_f1'] for r in gp2(ht,ratio,alpha)]
        for ht,ratio in HEADS
    ] + [[r['test']['macro_f1'] for r in gp3(alpha)]]

    bp = ax.boxplot(data_box, patch_artist=True, notch=False,
                    medianprops=dict(color='black',lw=2),
                    whiskerprops=dict(lw=1.2),
                    capprops=dict(lw=1.2),
                    flierprops=dict(marker='o',ms=4,alpha=0.5))
    for patch,color in zip(bp['boxes'],colors_b):
        patch.set_facecolor(color); patch.set_alpha(0.75)
    ax.set_xticklabels(labels_b,fontsize=9)
    ax.set_title(f'({chr(97+ai)}) '+ALPHA_STR[ai],fontweight='bold')
    ax.set_ylabel('Test Macro-F1' if ai==0 else '')
    ax.set_ylim(0.1,0.85)

fig.suptitle('Figure 4: Macro-F1 Distribution Across 5 Seeds',
             fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUT}/fig4_boxplots.pdf")
plt.savefig(f"{OUT}/fig4_boxplots.png",dpi=300)
plt.close(); print("Fig 4 saved.")

In [ ]:
# ── FIGURE 5: Per-class F1 ────────────────────────────────────────────
rows = gp2('bottleneck',4,1000)
pc   = np.array([r['test']['f1_per_class'] for r in rows])
mu   = pc.mean(0); sd = pc.std(0)
full = {'AKIEC':'Actinic Keratosis','BCC':'Basal Cell Carcinoma',
        'BKL':'Benign Keratosis','DF':'Dermatofibroma',
        'MEL':'Melanoma','NV':'Melanocytic Nevi','VASC':'Vascular'}

fig, ax = plt.subplots(figsize=(9,4))
x = np.arange(7)
ax.bar(x,mu,yerr=sd,color=C['fedavg'],alpha=0.80,capsize=5,width=0.55)
for xi,(m,s) in enumerate(zip(mu,sd)):
    ax.text(xi,m+s+0.015,f'{m:.3f}',ha='center',fontsize=9,fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES,fontsize=11)
ax.set_ylabel('Macro-F1 Score'); ax.set_ylim(0,1.05)
ax.set_xlabel('Skin Lesion Class')
ax.set_title('Figure 5: Per-Class F1 — Bottleneck N/4, '
             r'$\alpha$=1000 (mean±std, 5 seeds)',fontweight='bold')
ax2 = ax.twiny(); ax2.set_xlim(ax.get_xlim())
ax2.set_xticks(x)
ax2.set_xticklabels([full[c] for c in CLASS_NAMES],
                    rotation=30,ha='left',fontsize=8)
ax2.spines['top'].set_visible(False)
plt.tight_layout()
plt.savefig(f"{OUT}/fig5_per_class_f1.pdf")
plt.savefig(f"{OUT}/fig5_per_class_f1.png",dpi=300)
plt.close(); print("Fig 5 saved.")

In [ ]:
# ── FIGURE 6: Communication vs Accuracy ──────────────────────────────
f1_1000 = [np.mean([r['test']['macro_f1'] for r in gp2(ht,ratio,1000)])
           for ht,ratio in HEADS]
f1_01   = [np.mean([r['test']['macro_f1'] for r in gp2(ht,ratio,0.1)])
           for ht,ratio in HEADS]
pg_1000 = np.mean([r['test']['macro_f1'] for r in gp3(1000)])
pg_01   = np.mean([r['test']['macro_f1'] for r in gp3(0.1)])

fig, axes = plt.subplots(1,2,figsize=(10,4.5))
offsets = [(0.5,0.005),(0.5,0.005),(-8,0.005),(0.5,0.005)]

for ai,(f1s,pg_f1,atitle) in enumerate([
    (f1_1000,pg_1000,r'(a) $\alpha$=1000 (IID)'),
    (f1_01,  pg_01,  r'(b) $\alpha$=0.1 (non-IID)')]):
    ax = axes[ai]
    hcolors = [C['linear'],C['mlp'],C['bn4'],C['bn8']]
    for j,(label,mb,f1,hc) in enumerate(zip(HEAD_LABELS,MB_ROUND,f1s,hcolors)):
        ax.scatter(mb,f1,s=150,color=hc,zorder=5,edgecolors='black',lw=0.8)
        ox,oy = offsets[j]
        ax.annotate(label,(mb,f1),xytext=(mb+ox,f1+oy),fontsize=9,fontweight='bold')
    # PGA star at N/4 MB
    ax.scatter(3.581,pg_f1,s=220,color=C['pga'],marker='*',zorder=6,
               edgecolors='black',lw=0.8,label='PGA (Ours)')
    ax.annotate('PGA',(3.581,pg_f1),xytext=(25.5,pg_f1+0.005),
                fontsize=9,color=C['pga'],fontweight='bold')
    ax.set_xlabel('Uplink Communication (MB/round)')
    ax.set_ylabel('Test Macro-F1')
    ax.set_title(atitle,fontweight='bold')
    ax.set_xlim(-2,90)
    if ai==0: ax.legend()

fig.suptitle('Figure 6: Communication-Accuracy Trade-off',
             fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUT}/fig6_communication.pdf")
plt.savefig(f"{OUT}/fig6_communication.png",dpi=300)
plt.close(); print("Fig 6 saved.")

In [ ]:
# ── FIGURE 7: Scalability K=4,10,20 ──────────────────────────────────
fig, axes = plt.subplots(1,3,figsize=(12,4),sharey=False)

for ai,alpha in enumerate(ALPHAS):
    ax = axes[ai]
    Ks = [4,10]
    fa_m=[np.mean([r['test']['macro_f1'] for r in gp2('bottleneck',4,alpha)])]
    fa_s=[np.std ([r['test']['macro_f1'] for r in gp2('bottleneck',4,alpha)])]
    pg_m=[np.mean([r['test']['macro_f1'] for r in gp3(alpha)])]
    pg_s=[np.std ([r['test']['macro_f1'] for r in gp3(alpha)])]

    fa10=[r['test']['macro_f1'] for r in k10
          if r.get('method')=='fedavg' and r.get('alpha')==alpha]
    pg10=[r['test']['macro_f1'] for r in k10
          if r.get('method')=='pga'    and r.get('alpha')==alpha]
    if fa10: fa_m.append(np.mean(fa10)); fa_s.append(np.std(fa10))
    if pg10: pg_m.append(np.mean(pg10)); pg_s.append(np.std(pg10))

    fa20=[r['test']['macro_f1'] for r in k20
          if r.get('method')=='fedavg' and r.get('alpha')==alpha]
    pg20=[r['test']['macro_f1'] for r in k20
          if r.get('method')=='pga'    and r.get('alpha')==alpha]
    if fa20 and pg20:
        Ks.append(20)
        fa_m.append(np.mean(fa20)); fa_s.append(np.std(fa20))
        pg_m.append(np.mean(pg20)); pg_s.append(np.std(pg20))

    Kp = Ks[:len(fa_m)]
    ax.errorbar(Kp,fa_m[:len(Kp)],fa_s[:len(Kp)],
                marker='o',color=C['fedavg'],label='FedAvg',capsize=4)
    ax.errorbar(Kp,pg_m[:len(Kp)],pg_s[:len(Kp)],
                marker='s',color=C['pga'],label='PGA',capsize=4)
    ax.set_xlabel('Number of Clients (K)')
    ax.set_ylabel('Test Macro-F1' if ai==0 else '')
    ax.set_title(f'({chr(97+ai)}) '+ALPHA_STR[ai],fontweight='bold')
    ax.set_xticks(Kp); ax.set_ylim(0.15,0.80)
    if ai==0: ax.legend()

fig.suptitle('Figure 7: Scalability — FedAvg vs PGA across K Clients',
             fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUT}/fig7_scalability.pdf")
plt.savefig(f"{OUT}/fig7_scalability.png",dpi=300)
plt.close(); print("Fig 7 saved.")

In [ ]:
# ── FIGURE 8: Reliability diagrams (calibration) ─────────────────────
fig, axes = plt.subplots(1,2,figsize=(10,4.5))
rows_1000 = [r for r in p5 if r['alpha']==1000]

for pi,(key,title) in enumerate([
    ('reliability_no_scaling',  '(a) Before calibration (T=1)'),
    ('reliability_federated_T', '(b) After federated temperature scaling')]):

    ax = axes[pi]
    all_a=[]; all_c=[]
    for r in rows_1000:
        rel=r[key]; centres,accs,confs,counts=rel
        all_a.append(accs); all_c.append(confs)

    ma=np.mean(all_a,0); mc_=np.mean(all_c,0)
    centres=np.array(centres)
    mask=np.array([c>0 for c in mc_])

    ax.plot([0,1],[0,1],'k--',lw=1.2,label='Perfect calibration',alpha=0.7)
    ax.bar(centres[mask],ma[mask],width=0.065,alpha=0.55,
           color=C['pga'],edgecolor='white',lw=0.5)
    ax.plot(mc_[mask],ma[mask],'o-',color=C['fedavg'],lw=1.5,ms=5,
            label='Mean accuracy per bin')

    ece=[r['no_scaling' if pi==0 else 'federated_T']['ECE_test']
         for r in rows_1000]
    ax.text(0.05,0.88,f'ECE = {np.mean(ece):.4f}',
            transform=ax.transAxes,fontsize=10,
            bbox=dict(boxstyle='round,pad=0.3',facecolor='wheat',alpha=0.8))
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_xlabel('Confidence')
    ax.set_ylabel('Accuracy' if pi==0 else '')
    ax.set_title(title,fontweight='bold')
    ax.legend(fontsize=9,loc='upper left')

fig.suptitle(r'Figure 8: Reliability Diagrams — $\alpha$=1000 (mean, 5 seeds)',
             fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUT}/fig8_calibration.pdf")
plt.savefig(f"{OUT}/fig8_calibration.png",dpi=300)
plt.close(); print("Fig 8 saved.")

In [ ]:
# ── FIGURE 9: ISIC 2019 cross-dataset comparison ─────────────────────
fig, axes = plt.subplots(1,2,figsize=(11,4.5))

# Left: bar chart FedAvg vs PGA on ISIC 2019
ax = axes[0]
x=np.arange(3); w=0.3
fa_means=[np.mean([r['test']['macro_f1'] for r in isic_fa if r['alpha']==a])
          for a in ALPHAS]
fa_stds =[np.std ([r['test']['macro_f1'] for r in isic_fa if r['alpha']==a])
          for a in ALPHAS]
pg_means=[np.mean([r['test']['macro_f1'] for r in isic_pg if r['alpha']==a])
          for a in ALPHAS]
pg_stds =[np.std ([r['test']['macro_f1'] for r in isic_pg if r['alpha']==a])
          for a in ALPHAS]

ax.bar(x-w/2,fa_means,w,yerr=fa_stds,label='FedAvg',
       color=C['fedavg'],alpha=0.85,capsize=4)
ax.bar(x+w/2,pg_means,w,yerr=pg_stds,label='PGA (Ours)',
       color=C['pga'],alpha=0.85,capsize=4)
ax.set_xticks(x); ax.set_xticklabels(ALPHA_STR)
ax.set_ylabel('Test Macro-F1')
ax.set_title('(a) ISIC 2019 — FedAvg vs PGA',fontweight='bold')
ax.legend(); ax.set_ylim(0,0.9)

# Right: cross-dataset delta comparison
ax2 = axes[1]
ham_d  = [0.0,  0.004, 0.074]  # HAM10000 PGA-FedAvg delta per alpha
isic_d = [0.007,-0.008, 0.121]  # ISIC2019 PGA-FedAvg delta per alpha

x=np.arange(3); w=0.3
bars1=ax2.bar(x-w/2,ham_d, w,label='HAM10000',color='#2166AC',alpha=0.85)
bars2=ax2.bar(x+w/2,isic_d,w,label='ISIC 2019',color='#D6604D',alpha=0.85)
ax2.axhline(0,color='black',lw=0.8,linestyle='-')
ax2.set_xticks(x); ax2.set_xticklabels(ALPHA_STR)
ax2.set_ylabel('PGA − FedAvg (Macro-F1)')
ax2.set_title('(b) Cross-Dataset: PGA Improvement over FedAvg',
              fontweight='bold')
ax2.legend()

# Annotate significance
ax2.text(2-w/2,0.076,'**',ha='center',fontsize=13)
ax2.text(2+w/2,0.124,'ns',ha='center',fontsize=10,color='gray')

fig.suptitle('Figure 9: Cross-Dataset Generalizability (HAM10000 vs ISIC 2019)',
             fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUT}/fig9_cross_dataset.pdf")
plt.savefig(f"{OUT}/fig9_cross_dataset.png",dpi=300)
plt.close(); print("Fig 9 saved.")

In [ ]:
# ── ALL 7 TABLES in LaTeX format ──────────────────────────────────────
print("\n"+"="*70)
print("7 TABLES — LaTeX format for paper")
print("="*70)

# TABLE 1: Head ablation
print("\n--- TABLE 1: Head Ablation (HAM10000, alpha=1000, 5 seeds) ---")
print("\\begin{tabular}{lcccc}")
print("\\hline")
print("Head & Macro-F1 & Accuracy & Best Round & MB/round \\\\")
print("\\hline")
for i,(ht,ratio) in enumerate(HEADS):
    r=gp2(ht,ratio,1000)
    f1s=[x['test']['macro_f1'] for x in r]
    acs=[x['test']['accuracy']  for x in r]
    rds=[x['best_round']        for x in r]
    print(f"{HEAD_LABELS[i]} & ${np.mean(f1s):.3f}\\pm{np.std(f1s):.3f}$ & "
          f"${np.mean(acs):.3f}\\pm{np.std(acs):.3f}$ & "
          f"${np.mean(rds):.1f}\\pm{np.std(rds):.1f}$ & {MB_ROUND[i]} \\\\")
print("\\hline\n\\end{tabular}")

# TABLE 2: Heterogeneity
print("\n--- TABLE 2: Heterogeneity Sweep ---")
print("\\begin{tabular}{lcccc}")
print("\\hline")
print("Head & $\\alpha$=1000 & $\\alpha$=1 & $\\alpha$=0.1 & Drop\\\\")
print("\\hline")
for i,(ht,ratio) in enumerate(HEADS):
    vals=[(np.mean([r['test']['macro_f1'] for r in gp2(ht,ratio,a)]),
           np.std( [r['test']['macro_f1'] for r in gp2(ht,ratio,a)])) for a in ALPHAS]
    drop=vals[0][0]-vals[2][0]
    print(f"{HEAD_LABELS[i]} & ${vals[0][0]:.3f}\\pm{vals[0][1]:.3f}$ & "
          f"${vals[1][0]:.3f}\\pm{vals[1][1]:.3f}$ & "
          f"${vals[2][0]:.3f}\\pm{vals[2][1]:.3f}$ & {drop:.3f} \\\\")
print("\\hline\n\\end{tabular}")

# TABLE 3: FedAvg vs FedProx vs PGA
print("\n--- TABLE 3: FedAvg vs FedProx vs PGA (10 seeds at alpha=0.1) ---")
fa04=[r['test']['macro_f1'] for r in gp2('bottleneck',4,0.1)]
fp04=[r['test']['macro_f1'] for r in gfp(0.1)]
pg04=[r['test']['macro_f1'] for r in gp3(0.1)]
all_fa=fa04+fa_59; all_fp=fp04+fp_59; all_pg=pg04+pga_59
_,pv1=ttest_rel(all_pg,all_fa); d1=np.mean(all_pg)-np.mean(all_fa)
_,pv2=ttest_rel(all_pg,all_fp); d2=np.mean(all_pg)-np.mean(all_fp)
ci1=1.96*np.std(np.array(all_pg)-np.array(all_fa))/np.sqrt(10)
ci2=1.96*np.std(np.array(all_pg)-np.array(all_fp))/np.sqrt(10)
print("\\begin{tabular}{lcccc}")
print("\\hline")
print("Method & $\\alpha$=1000 & $\\alpha$=1 & $\\alpha$=0.1 (10 seeds) & Sig.\\\\")
print("\\hline")
for method,label,src in [('fedavg','FedAvg','p2'),
                           ('fedprox','FedProx','fp'),
                           ('pga','\\textbf{PGA (Ours)}','p3')]:
    vals=[]
    for alpha in ALPHAS:
        if src=='p2': rows=gp2('bottleneck',4,alpha)
        elif src=='p3': rows=gp3(alpha)
        else: rows=gfp(alpha)
        f1s=[r['test']['macro_f1'] for r in rows]
        vals.append(f"${np.mean(f1s):.3f}\\pm{np.std(f1s):.3f}$")
    sig='—' if method=='fedavg' else         f"vs FA: {sig_str(pv1)}" if method=='pga' else         f"vs FA: {sig_str(ttest_rel(fp04,fa04)[1])}"
    print(f"{label} & {vals[0]} & {vals[1]} & {vals[2]} & {sig} \\\\")
print(f"\\multicolumn{{5}}{{l}}{{PGA vs FedAvg: $\\Delta$={d1:+.4f}, "
      f"95\\%CI[{d1-ci1:+.4f},{d1+ci1:+.4f}], p={pv1:.4f}}}")
print("\\hline\n\\end{tabular}")

# TABLE 4: Scalability
print("\n--- TABLE 4: Scalability ---")
print("\\begin{tabular}{llccc}")
print("\\hline")
print("Method & K & $\\alpha$=1000 & $\\alpha$=1 & $\\alpha$=0.1\\\\")
print("\\hline")
for method,label,src in [('fedavg','FedAvg','p2'),('pga','PGA','p3')]:
    for K in [4,10,20]:
        vals=[]
        for alpha in ALPHAS:
            if K==4:
                rows=gp2('bottleneck',4,alpha) if src=='p2' else gp3(alpha)
            elif K==10:
                rows=[r for r in k10 if r.get('method')==method and r.get('alpha')==alpha]
            else:
                rows=[r for r in k20 if r.get('method')==method and r.get('alpha')==alpha]
            if rows:
                f1s=[r['test']['macro_f1'] for r in rows]
                vals.append(f"${np.mean(f1s):.3f}\\pm{np.std(f1s):.3f}$")
            else: vals.append("—")
        print(f"{label} & {K} & {vals[0]} & {vals[1]} & {vals[2]} \\\\")
    print("\\hline")
print("\\end{tabular}")

# TABLE 5: CV
print("\n--- TABLE 5: 5-fold Cross-Validation ---")
print("\\begin{tabular}{lccc}")
print("\\hline")
print("Method & Val Macro-F1 & Test Macro-F1 & p-value\\\\")
print("\\hline")
fa_cv=[r['test']['macro_f1'] for r in cv if r['method']=='fedavg']
pg_cv=[r['test']['macro_f1'] for r in cv if r['method']=='pga']
_,pv_cv=ttest_rel(pg_cv,fa_cv)
for method,label in [('fedavg','FedAvg'),('pga','PGA (Ours)')]:
    vf=[r['val_f1'] for r in cv if r['method']==method]
    tf=[r['test']['macro_f1'] for r in cv if r['method']==method]
    sig_cv=sig_str(pv_cv) if method=='pga' else '—'
    print(f"{label} & ${np.mean(vf):.3f}\\pm{np.std(vf):.3f}$ & "
          f"${np.mean(tf):.3f}\\pm{np.std(tf):.3f}$ & {sig_cv} \\\\")
print("\\hline\n\\end{tabular}")

# TABLE 6: ISIC 2019
print("\n--- TABLE 6: ISIC 2019 Cross-Dataset ---")
print("\\begin{tabular}{lcccc}")
print("\\hline")
print("$\\alpha$ & FedAvg & PGA & $\\Delta$ & p-value\\\\")
print("\\hline")
for alpha in ALPHAS:
    fa=[r['test']['macro_f1'] for r in isic_fa if r['alpha']==alpha]
    pg=[r['test']['macro_f1'] for r in isic_pg if r['alpha']==alpha]
    if fa and pg:
        _,pv=ttest_rel(pg,fa); d=np.mean(pg)-np.mean(fa)
        print(f"{alpha} & ${np.mean(fa):.3f}\\pm{np.std(fa):.3f}$ & "
              f"${np.mean(pg):.3f}\\pm{np.std(pg):.3f}$ & "
              f"${d:+.4f}$ & {pv:.4f}{sig_str(pv)} \\\\")
print("\\hline\n\\end{tabular}")

# TABLE 7: Calibration
print("\n--- TABLE 7: Federated Temperature Scaling ---")
print("\\begin{tabular}{lcccc}")
print("\\hline")
print("$\\alpha$ & ECE (none) & ECE (global T) & ECE (fed T) & Reduction\\\\")
print("\\hline")
for alpha in ALPHAS:
    rows=[r for r in p5 if r['alpha']==alpha]
    ns=np.mean([r['no_scaling'  ]['ECE_test'] for r in rows])
    gt=np.mean([r['global_T'   ]['ECE_test'] for r in rows])
    ft=np.mean([r['federated_T']['ECE_test'] for r in rows])
    _,pv=ttest_rel([r['no_scaling']['ECE_test'] for r in rows],
                   [r['federated_T']['ECE_test'] for r in rows])
    print(f"{alpha} & {ns:.4f} & {gt:.4f} & {ft:.4f} & "
          f"{(ns-ft)/ns*100:.1f}\\%{sig_str(pv)} \\\\")
print("\\hline\n\\end{tabular}")

print("\n"+"="*70)
print(f"ALL 9 FIGURES + 7 TABLES COMPLETE")
files = sorted(os.listdir(OUT))
print(f"Files in {OUT}:")
for f in files:
    sz=os.path.getsize(os.path.join(OUT,f))//1024
    print(f"  {f}  ({sz} KB)")